In [126]:
"""
model_generation.py
Run this script ONCE to generate the model files (movie_list.pkl and similarity.pkl).
Place tmdb_5000_movies.csv and tmdb_5000_credits.csv in the same folder before running.
"""

import numpy as np
import pandas as pd
import ast
import pickle
import os
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ─── 1. Load Data ────────────────────────────────────────────────────────────

BASE_DIR = os.getcwd()

movies  = pd.read_csv(os.path.join(BASE_DIR, 'tmdb_5000_movies.csv'))
credits = pd.read_csv(os.path.join(BASE_DIR, 'tmdb_5000_credits.csv'))

# ─── 2. Merge & Select Relevant Columns ──────────────────────────────────────

movies = movies.merge(credits, on='title')
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)

# ─── 3. Helper Functions ──────────────────────────────────────────────────────

def convert(text):
    """Extract all 'name' fields from a JSON-like string."""
    return [i['name'] for i in ast.literal_eval(text)]

def convert_cast(text):
    """Extract top-3 cast names."""
    return [i['name'] for i in ast.literal_eval(text)][:3]

def fetch_director(text):
    """Extract director name from crew."""
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

def collapse(L):
    """Remove spaces so 'Sam Mendes' → 'SamMendes' (avoids false matches)."""
    return [i.replace(" ", "") for i in L]

# ─── 4. Feature Engineering ───────────────────────────────────────────────────

movies['genres']   = movies['genres'].apply(convert).apply(collapse)
movies['keywords'] = movies['keywords'].apply(convert).apply(collapse)
movies['cast']     = movies['cast'].apply(convert_cast).apply(collapse)
movies['crew']     = movies['crew'].apply(fetch_director).apply(collapse)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Combine all features into a single 'tags' column
movies['tags'] = (
    movies['overview'] +
    movies['genres']   +
    movies['keywords'] +
    movies['cast']     +
    movies['crew']
)

# ─── 5. Build Final DataFrame ────────────────────────────────────────────────

new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x)).str.lower()

# ─── 6. Vectorize & Compute Similarity ───────────────────────────────────────

cv = CountVectorizer(max_features=5000, stop_words='english')
vector = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vector)

print(f"✅ Vectorized shape : {vector.shape}")
print(f"✅ Similarity matrix: {similarity.shape}")

# ─── 7. Quick Sanity Check ───────────────────────────────────────────────────

def recommend(movie):
    index = new_df[new_df['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])
    print(f"\nMovies similar to '{movie}':")
    for i in distances[1:6]:
        print(" -", new_df.iloc[i[0]].title)

recommend('The Avengers')

# ─── 8. Save Model Files ──────────────────────────────────────────────────────

model_dir = os.path.join(BASE_DIR, 'model')
os.makedirs(model_dir, exist_ok=True)

pickle.dump(new_df,      open(os.path.join(model_dir, 'movie_list.pkl'), 'wb'))
pickle.dump(similarity,  open(os.path.join(model_dir, 'similarity.pkl'), 'wb'))

print(f"\n✅ Model files saved to: {model_dir}")
print("   → model/movie_list.pkl")
print("   → model/similarity.pkl")

✅ Vectorized shape : (4806, 5000)
✅ Similarity matrix: (4806, 4806)

Movies similar to 'The Avengers':
 - Avengers: Age of Ultron
 - Captain America: Civil War
 - Iron Man 3
 - Captain America: The First Avenger
 - Iron Man

✅ Model files saved to: C:\Users\premc\Downloads\proj4MovieRecommendationSystem\model
   → model/movie_list.pkl
   → model/similarity.pkl
